# Phase X.X - Extraction d'aspects

Ce notebook intègre les approches fondées sur un modèle de langage

- **Extraction d’aspects (Aspect-Based Sentiment Analysis)** :
Une même revue peut être positive sur certains aspects et négative sur d’autres
(par exemple pour un hôtel : lit, bruit, propreté ; pour un restaurant : nourriture,
service, prix).

## 0. Préparation des modèles

Le model choisie : Llama3.2

La collection Meta Llama 3.2 de grands modèles de langage multilingues (LLM) est un ensemble de modèles génératifs pré-entraînés et optimisés pour les instructions, disponibles en tailles 1 et 3 milliards (texte en entrée/texte en sortie). Les modèles Llama 3.2 optimisés pour le texte uniquement sont conçus pour les cas d'utilisation de dialogues multilingues, notamment la recherche et la synthèse automatiques.

In [1]:
import sys
from langchain_ollama import ChatOllama
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_local_llm(model_name="llama3.2"):
    """
    Configure et retourne une instance du modèle local via Ollama.
    """
    print(f"Connexion au modèle local '{model_name}' via Ollama...")

    try:
        llm = ChatOllama(
            model=model_name,
            temperature=0,
            callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
        )
        
        return llm

    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE : Impossible de joindre Ollama.")
        print(f"Détails : {e}")
        print("-" * 40)
        return None

c:\Users\cleme\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Prendre des "reviews" pour tests

In [2]:
import json
import random

reviews_file = "../data/raw/yelp_academic_reviews4students.jsonl"

reviews = []
with open(reviews_file, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))


# Prendre 10 reviews aléatoires parmi les filtrées
sample_reviews = random.sample(reviews, min(1, len(reviews)))


---

In [3]:
import pandas as pd

absa_results = []

llm = get_local_llm()

Connexion au modèle local 'llama3.2' via Ollama...


## 1. Analyse

In [4]:
from langchain_core.prompts import PromptTemplate
import json

# Template conçu pour l'extraction structurée (Partie 3 du sujet)
template_absa = """
Tu es un expert en analyse de données Yelp spécialisé en Aspect-Based Sentiment Analysis (ABSA).

TÂCHE :
Analyse l'avis client fourni et extrais les aspects mentionnés UNIQUEMENT s'ils sont associés à un sentiment explicite. 
Tu dois mapper chaque aspect identifié vers une CATEGORIES GENERALES.

CONTRAINTES :
1. Sentiment : Doit être EXACTEMENT "positif" ou "négatif".
2. Exclusion : Si un aspect est mentionné sans sentiment clair ou est ambigu, NE L'INCLUS PAS.
3. Format : Retourne STRICTEMENT un objet JSON (une liste d'objets). Ne rajoute aucune phrase avant ou après le JSON.

Format de réponse :
[
  {{"aspect": "Nom de la catégorie générale", "sentiment": "positif/négatif"}}
]

Si aucun aspect clair n'est trouvé, retourne : []

AVIS À ANALYSER :
{texte_avis}

RÉPONSE JSON :
"""

prompt_absa = PromptTemplate.from_template(template_absa)
chain_absa = prompt_absa | llm

print("🚀 Lancement de l'extraction d'aspects (ABSA)...")

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    
    response = chain_absa.invoke({"texte_avis": text})
    content = response.content
    
    try:
        # Nettoyage de la réponse pour ne garder que le JSON
        json_start = content.find('[')
        json_end = content.rfind(']') + 1
        aspects = json.loads(content[json_start:json_end])
        
        for item in aspects:
            aspect = item.get('aspect', '').strip()
            sentiment = item.get('sentiment', '').strip().lower()
            
            # Validation stricte : accepter UNIQUEMENT "positif" ou "négatif"
            if sentiment in ['positif', 'négatif'] and aspect:
                absa_results.append({
                    "id_review": i,
                    "aspect": aspect,
                    "sentiment": sentiment
                })
            
    except Exception as e:
        print(f"⚠️ Erreur d'extraction sur l'avis {i}: {e}")

df_absa = pd.DataFrame(absa_results)
display(df_absa.head(10))

🚀 Lancement de l'extraction d'aspects (ABSA)...


,id_review,aspect,sentiment
0,1,Qualité de la nourriture,positif
1,1,Service,positif
2,1,Portion,positif
3,1,Noodles,négatif


## 3. Résultat

In [5]:
# Affichage détaillé par avis
print("Détails par avis :")
for review_id in df_absa['id_review'].unique():
    print(f"\n{'='*100}")
    print(f"📌 AVIS #{review_id}")
    
    # Afficher le texte complet de l'avis
    review_text = sample_reviews[review_id - 1]['text']
    print(f"\n📝 TEXTE COMPLET :")
    print(f"   {review_text}\n")
    
    # Afficher les aspects identifiés
    print(f"📊 ASPECTS IDENTIFIÉS :")
    subset = df_absa[df_absa['id_review'] == review_id]
    
    if len(subset) == 0:
        print(f"   ℹ️  Aucun aspect clair identifié")
    else:
        for _, row in subset.iterrows():
            sentiment = row['sentiment'].lower()
            sentiment_symbol = "✅" if sentiment == 'positif' else "❌"
            print(f"   {sentiment_symbol} {row['aspect']}: {sentiment}")

print(f"\n{'='*100}")

Détails par avis :

📌 AVIS #1

📝 TEXTE COMPLET :
   This place was huge. Anyone know what it used to be? Anyway, we cane here for dinner one someone chilly night looking for so ramen. We've been getting spoiled here with Kin and Union opening in the last few years. (So sad Kin is closed for good now). Anyway, we ordse some sushi to start and that rice was perfection. I really enjoyed the sushi alot. I also order the hangout ichimi ramen. I was going to order the heat as burning but I stuck with mild when the waitress said burning was too hot for her. There are three levels of heat. It comes with pork belly and that was tender and flavorful! The shrimp were fried on a stick, just thought that was oddly served that way. I guess I thought it would be sautéed or boiled shrimp in the ramen. The egg unfortunately did not have much soaked in broth flavors. I am guessing they don't cook the egg with the broth. The broth itself had great flavor and some heat so I can see why I may never be able